In [2]:
using Base.Threads


results = zeros(100)
Threads.@threads for i in 1:100
    results[i] = i^2
    println("Computed for $i on thread $(threadid())")
end

println("Results: $results")

Computed for 51 on thread 3
Computed for 26 on thread 1
Computed for 52 on thread 3
Computed for 76 on thread 4
Computed for 77 on thread 2
Computed for 53 on thread 3
Computed for 78 on thread 2
Computed for 54 on thread 3
Computed for 55 on thread 3
Computed for 1 on thread 2
Computed for 79 on thread 2
Computed for 56 on thread 3
Computed for 27 on thread 4
Computed for 2 on thread 4
Computed for 28 on thread 2
Computed for 29 on thread 2
Computed for 3 on thread 4
Computed for 80 on thread 2
Computed for 81 on thread 2
Computed for 82 on thread 2
Computed for 30 on thread 2
Computed for 83 on thread 2
Computed for 57 on thread 4
Computed for 4 on thread 3
Computed for 84 on thread 3
Computed for 31 on thread 4
Computed for 5 on thread 2
Computed for 32 on thread 2
Computed for 6 on thread 4
Computed for 33 on thread 2
Computed for 7 on thread 4
Computed for 58 on thread 4
Computed for 34 on thread 2
Computed for 59 on thread 3
Computed for 8 on thread 3
Computed for 85 on thread 3


In [3]:
using Base.Threads


function block_multiply!(C, A, B, block_size)
    n = size(A, 1)  # Square matrices

    #for ii in 1:block_size:n # single-threaded
    @threads for ii in 1:block_size:n # multi-threaded
        for jj in 1:block_size:n
            for kk in 1:block_size:n
                for i in ii:min(ii + block_size - 1, n)
                    for j in jj:min(jj + block_size - 1, n)
                        for k in kk:min(kk + block_size - 1, n)
                            C[i, j] += A[i, k] * B[k, j]
                        end
                    end
                end
            end
        end
    end
end


function test_multithreaded_matmul()
    n = 1024  # Matrix dimensions n^2
    block_size = 64  # Decomposition block size

    A = randn(n, n)
    B = randn(n, n)
    C = zeros(n, n)

    println("Number of threads: $(nthreads())")
    println("Starting block matrix multiplication...")

    @time block_multiply!(C, A, B, block_size)

    println("Computation completed.")
    println("Checksum of result matrix: ", sum(C))
end


test_multithreaded_matmul()

Number of threads: 4
Starting block matrix multiplication...
  6.705396 seconds (66.46 k allocations: 3.366 MiB, 4.85% compilation time)
Computation completed.
Checksum of result matrix: 14856.470966620349


In [5]:
using LibGEOS

# 1. Define geometries
# A Point requires a single coordinate pair (x, y)
point_internal = LibGEOS.Point(5.0, 5.0)
point_external = LibGEOS.Point(15.0, 15.0)

# A Polygon requires an array of rings. 
# The first ring represents the exterior boundary.
exterior_ring = [[0.0, 0.0], [10.0, 0.0], [10.0, 10.0], [0.0, 10.0], [0.0, 0.0]]
polygon = LibGEOS.Polygon([exterior_ring])

# 2. Execute spatial operations
# Calculate the total area of the polygon
poly_area = LibGEOS.area(polygon)
println("Polygon Area: ", poly_area)

# Evaluate topological relationships
is_contained = LibGEOS.contains(polygon, point_internal)
println("Does the polygon contain the internal point? ", is_contained)

# Calculate the minimum distance between two geometries
distance_to_poly = LibGEOS.distance(polygon, point_external)
println("Distance from the external point to the polygon: ", distance_to_poly)

# 3. Generate new geometries
# Create a buffer around a point (results in a polygon)
buffered_point = LibGEOS.buffer(point_internal, 2.0)
println("Area of the buffered point: ", LibGEOS.area(buffered_point))

Polygon Area: 100.0
Does the polygon contain the internal point? true
Distance from the external point to the polygon: 7.0710678118654755
Area of the buffered point: 12.485780609032208


In [6]:
using GeoJSON

# 1. Define a GeoJSON string
# This example uses a Feature representing the coordinates for London
geojson_input = """
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [-0.1278, 51.5074]
  },
  "properties": {
    "city": "London",
    "country": "United Kingdom",
    "population": 8982000
  }
}
"""

# 2. Parse the GeoJSON string into a Julia object
feature = GeoJSON.read(geojson_input)
println("Successfully parsed the GeoJSON feature.")

# 3. Access properties and geometry
# Properties can typically be accessed directly via dot notation
city_name = feature.city
population = feature.population

println("City: ", city_name)
println("Population: ", population)

# Access the underlying geometry coordinates
geometry = GeoJSON.geometry(feature)
coords = GeoJSON.coordinates(geometry)
println("Longitude: ", coords[1], ", Latitude: ", coords[2])

# 4. Modify properties and write back to a GeoJSON string
# Note: To modify properties, you often construct a new Dict or Feature 
# depending on the specific version of GeoJSON.jl. Here, we demonstrate 
# writing the existing object back to a string format.
output_string = GeoJSON.write(feature)
println("\nGenerated GeoJSON output:")
println(output_string)

Successfully parsed the GeoJSON feature.
City: London
Population: 8982000
Longitude: -0.1278, Latitude: 51.5074

Generated GeoJSON output:
{"type":"Feature","geometry":{"type":"Point","coordinates":[-0.1278,51.5074]},"properties":{"country":"United Kingdom","population":8982000,"city":"London"}}


In [15]:
using ZarrDatasets
using Statistics

# Define the remote URL for the GPCP dataset
remote_zarr_url = "https://ncsa.osn.xsede.org/Pangeo/pangeo-forge/gpcp-feedstock/gpcp.zarr"

ZarrDataset(remote_zarr_url) do ds
    println("Successfully connected to the remote Zarr store.")
    
    # Construct a single string from the keys to circumvent IJulia array display issues
    available_vars_string = join(keys(ds), ", ")
    println("Available variables are: ", available_vars_string)
    
    # Verify and extract the target variable
    if haskey(ds, "precip")
        precip_var = ds["precip"]
        
        println("\nDownloading spatial data for the first time step...")
        # Download the spatial slice for the first time step
        precip_spatial_slice = precip_var[:, :, 1]
        
        # Circumvent the tuple display error by formatting the dimensions as a string explicitly
        dims = size(precip_spatial_slice)
        dims_string = join(dims, " by ")
        println("Download complete. Array size: ", dims_string)
        
        # Filter out missing values representing invalid geographic areas
        valid_precipitation = collect(skipmissing(precip_spatial_slice))
        
        if isempty(valid_precipitation)
            println("No valid data points were found in this specific slice.")
        else
            mean_precip = mean(valid_precipitation)
            max_precip = maximum(valid_precipitation)
            
            println("\n--- Data Analysis Results ---")
            println("Valid data points processed: ", length(valid_precipitation))
            println("Mean Precipitation:    ", round(mean_precip, digits=2))
            println("Maximum Precipitation: ", round(max_precip, digits=2))
        end
    else
        println("The requested variable 'precip' was not found.")
    end
end

Successfully connected to the remote Zarr store.
Available variables are: latitude, time, longitude, time_bounds, lon_bounds, precip, lat_bounds

Download complete. Array size: 360 by 180

--- Data Analysis Results ---
Valid data points processed: 64800
Mean Precipitation:    2.33
Maximum Precipitation: 99.59
